# Sensibilidad de los umbrales de accesibilidad en Sonora

Se usan 60 minutos para centros de salud y 120 para hospitales. Se compara la clasificación estricta `tiempo > umbral` al variar ambos cortes ±15 minutos. Un tiempo faltante queda fuera de la comparación. La población se cuenta por nivel y no se suma entre niveles.

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd

raiz = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/health_data").is_dir())
sys.path.insert(0, str(raiz / "src"))

from health_data.calidad_ruteo import tiempos_por_nivel, sensibilidad

procesados = Path(os.environ.get("HEALTH_DATA_PROCESSED_DIR", raiz / "data/processed"))
ruta = procesados / "accesibilidad_localidades.parquet"
tabla = pd.read_parquet(ruta)
tiempos = tiempos_por_nivel(tabla)
resumen, casos = sensibilidad(tiempos, margen_min=15)

medidas = ["localidades", "con_tiempo", "sobre_umbral", "cambia_umbral_menor",
           "cambia_umbral_mayor", "poblacion_cambia"]
display(resumen.groupby(["nivel", "metodo"])[medidas].sum())

localidades  con_tiempo  sobre_umbral  \
nivel         metodo                                           
Primer nivel  coneval           58          58            17   
              sakbe           7239        7239           517   
Segundo nivel coneval           56          56            36   
              sakbe           7241        7241           635   

                       cambia_umbral_menor  cambia_umbral_mayor  \
nivel         metodo                                              
Primer nivel  coneval                   19                    5   
              sakbe                    310                  188   
Segundo nivel coneval                    0                    3   
              sakbe                    212                  120   

                       poblacion_cambia  
nivel         metodo                     
Primer nivel  coneval               138  
              sakbe                5241  
Segundo nivel coneval                 6  
              sakbe               10190

## Municipios y localidades prioritarias

Se ordena el impacto municipal por población, y los casos individuales por población y cercanía al corte. Este orden facilita revisar las localidades con mayor población afectada.

In [2]:
display(resumen.sort_values("poblacion_cambia", ascending=False).head(25))
display(casos[["locality_id", "locality_name", "municipality_name", "nivel", "metodo",
               "population", "tiempo_min", "distancia_al_umbral_min",
               "cambia_umbral_menor", "cambia_umbral_mayor"]].head(30))

,nivel,metodo,municipality_name,localidades,con_tiempo,sobre_umbral,cambia_umbral_menor,cambia_umbral_mayor,poblacion_cambia
115,Segundo nivel,sakbe,Alamos,272,272,38,23,18,1619
51,Primer nivel,sakbe,Hermosillo,1018,1018,61,57,32,1458
125,Segundo nivel,sakbe,Bavispe,6,6,3,3,1,1101
122,Segundo nivel,sakbe,Bacerac,13,13,5,7,0,1085
165,Segundo nivel,sakbe,Rosario,72,72,49,10,10,848
149,Segundo nivel,sakbe,La Colorada,92,92,19,15,11,836
23,Primer nivel,sakbe,Alamos,272,272,22,14,12,666
121,Segundo nivel,sakbe,Bacanora,16,16,11,5,2,659
175,Segundo nivel,sakbe,Soyopa,40,40,16,11,4,541
166,Segundo nivel,sakbe,Sahuaripa,43,43,35,4,3,528


,locality_id,locality_name,municipality_name,nivel,metodo,population,tiempo_min,distancia_al_umbral_min,cambia_umbral_menor,cambia_umbral_mayor
7869,260100001,Bacerac,Bacerac,Segundo nivel,sakbe,1019,109.2,10.8,True,False
9698,260210061,Tecoripa,La Colorada,Segundo nivel,sakbe,738,127.6,7.6,False,True
8141,260150001,Bavispe,Bavispe,Segundo nivel,sakbe,674,119.3,0.7,True,False
7853,260090001,Bacanora,Bacanora,Segundo nivel,sakbe,498,110.1,9.9,True,False
7810,260060084,Sinoquipe,Arizpe,Segundo nivel,sakbe,390,111.7,8.3,True,False
8144,260150029,San Miguelito,Bavispe,Segundo nivel,sakbe,374,123.4,3.4,False,True
13930,260610018,La Estrella,Soyopa,Segundo nivel,sakbe,351,105.8,14.2,True,False
13126,260510047,Nuri,Rosario,Segundo nivel,sakbe,341,107.0,13.0,True,False
5906,260520044,Matarachi,Sahuaripa,Primer nivel,sakbe,334,60.9,0.9,False,True
5821,260510015,Cedros,Rosario,Primer nivel,sakbe,307,46.0,14.0,True,False


## Tiempo disponible y respaldo CONEVAL

Las categorías CONEVAL describen el tiempo a un centro de servicios. Sirven para estratificar el análisis. Los minutos imputados a un establecimiento requieren rutas SAKBÉ de referencia. Si existen tiempos SAKBÉ o imputados, se comparan con la estimación por distancia en línea recta ajustada al mismo CLUES que figura en la tabla. El respaldo CONEVAL usa una mediana de grupo, de modo que el CLUES solo sirve de referencia para el escenario alternativo.

In [3]:
display(tiempos.groupby(["nivel", "metodo", "tiempo_centro_servicios"],
                      dropna=False, observed=True)
        .agg(localidades=("nivel", "size"), poblacion=("population", "sum")))
if not tiempos["metodo"].eq("coneval").any():
    print("Este corte no contiene tiempos imputados con CONEVAL.")

from health_data.calidad_ruteo import comparar_con_estimacion

salud = procesados / "salud-001_establecimientos.parquet"
if salud.is_file() and tiempos["metodo"].isin(["sakbe", "coneval"]).any():
    pares = comparar_con_estimacion(tabla, pd.read_parquet(salud))
    display(pares.groupby(["nivel", "metodo", "municipality_name"], dropna=False)
            .agg(pares=("nivel", "size"), cambios=("cambia_clasificacion", "sum"),
                 poblacion_cambia=("poblacion_cambia", "sum")))
else:
    print("No hay datos locales suficientes para comparar escenarios emparejados.")

localidades  poblacion
nivel         metodo  tiempo_centro_servicios                        
Primer nivel  coneval MENOS DE 1 HR.                     6         14
                      1 A 2 HRS.                         6         22
                      2 A 3 HRS.                        13         28
                      3 A 4 HRS.                        13         49
                      MAS DE 4 HRS.                     20        128
              sakbe   MENOS DE 1 HR.                  4596    2726693
                      1 A 2 HRS.                      1550     139127
                      2 A 3 HRS.                       639      45033
                      3 A 4 HRS.                       284      21968
                      MAS DE 4 HRS.                    170      11762
Segundo nivel coneval MENOS DE 1 HR.                     6         14
                      1 A 2 HRS.                         6         22
                      2 A 3 HRS.                        13         28
                      3 A 4 HRS.                        13         49
                      MAS DE 4 HRS.                     18        122
              sakbe   MENOS DE 1 HR.                  4596    2726693
                      1 A 2 HRS.                      1550     139127
                      2 A 3 HRS.                       639      45033
                      3 A 4 HRS.                       284      21968
                      MAS DE 4 HRS.                    172      11768

pares  cambios  poblacion_cambia
nivel         metodo  municipality_name                                  
Primer nivel  coneval Alamos                 3        2                 5
                      Altar                  2        2                 6
                      Arizpe                 1        0                 0
                      Baviácora              1        0                 0
                      Bácum                  1        0                 0
...                                        ...      ...               ...
Segundo nivel sakbe   Tubutama              47        5                76
                      Ures                 134        0                 0
                      Villa Hidalgo          3        1               245
                      Villa Pesqueira       36        0                 0
                      Yécora                67        0                 0

[185 rows x 3 columns]